# LeetCode Hot 100 - Day 15

## 今日主题：构造、路径与最大路径和

今天三道题都在练“递归函数返回什么”这件事：

1. 从前序与中序遍历序列构造二叉树：用两个遍历序列切成子问题。
2. 路径总和 III：双重递归，每个节点都当一次起点。
3. 二叉树中的最大路径和：和昨天的直径是同一个套路，但加了负数处理。

第 3 题是困难题，也是大厂高频，值得多花时间。


## 今天怎么学

1. 第一题先想清楚“怎么切”，切法对了代码就短。
2. 第二题重点理解“以每个节点为起点”的双重递归。
3. 第三题是昨天直径的升级版，注意“负数不如不要”这个处理。
4. 加练：把昨天的二叉树的直径、展开为链表重新默写一遍。

最低目标：独立写出从前序与中序遍历构造二叉树。


## 今日题单

1. 从前序与中序遍历序列构造二叉树（LeetCode 105，中等，必做）
2. 路径总和 III（LeetCode 437，中等，必做）
3. 二叉树中的最大路径和（LeetCode 124，困难，必做）
4. 加练：重写 Day 14 的二叉树的直径和二叉树展开为链表


## 昨日复习

先用 `day14_practice.ipynb` 重写：

1. 二叉树的直径：函数返回深度，顺路更新答案。
2. 二叉树的右视图：层序遍历取每层最后一个。

口述：为什么右视图不能只沿着 right 指针走？


## 今天的树工具

继续用昨天的三个工具，写法完全一致：

- `TreeNode`：节点定义。
- `build_tree(values)`：按层序数组建树。
- `tree_values(root)`：把树转回层序列表，方便对比结果。


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


root = build_tree([3, 9, 20, None, None, 15, 7])
print(tree_values(root))


## 题目 1 做题前先补：两种遍历各自能告诉你什么

先复习两条最重要的性质：

- **前序遍历**：顺序是 `根 -> 左子树 -> 右子树`。所以**第一个元素一定是根**。
- **中序遍历**：顺序是 `左子树 -> 根 -> 右子树`。所以**根把中序序列劈成两半**：左边是左子树的所有节点，右边是右子树的所有节点。

把这两条合起来，就能唯一确定一棵树：

1. 从前序里拿出第一个元素，作为根；
2. 在中序里找到这个根的位置 `mid`；
3. 中序的 `[0, mid)` 是左子树，`(mid, 末尾]` 是右子树；
4. 因为左子树有 `mid` 个节点，所以前序里 `[1, 1 + mid)` 是左子树，剩下的就是右子树；
5. 左右两边递归。

下面用一段代码把“切”的过程打印出来。


In [ ]:
preorder = [3, 9, 20, 15, 7]
inorder = [9, 3, 15, 20, 7]

root_val = preorder[0]
mid = inorder.index(root_val)

print("根是：", root_val)
print("在中序里的位置：", mid)
print("左子树的中序：", inorder[:mid], "，左子树的前序：", preorder[1:1 + mid])
print("右子树的中序：", inorder[mid + 1:], "，右子树的前序：", preorder[1 + mid:])


# 题目 1：从前序与中序遍历序列构造二叉树

LeetCode 105. Construct Binary Tree from Preorder and Inorder Traversal

## 题目描述（改写版）

给你两个整数数组 `preorder` 和 `inorder`：

- `preorder` 是某棵二叉树的**前序遍历**结果；
- `inorder` 是同一棵树的**中序遍历**结果；
- 树中所有节点的值**互不相同**。

请根据这两个序列把这棵树重新构造出来，返回根节点。

## 输入

- `preorder`、`inorder`：长度 1 到 3000，元素互不相同。

## 输出

返回构造出来的树的根节点。

## 示例

示例 1：`preorder = [3, 9, 20, 15, 7]`，`inorder = [9, 3, 15, 20, 7]`，构造出的树是 `[3, 9, 20, None, None, 15, 7]`。

示例 2：`preorder = [-1]`，`inorder = [-1]`，只有一个节点。

## 易漏细节

- 前序的第一个元素是根，中序里根的位置决定左右子树的大小。
- 左子树的前序是 `preorder[1:1 + mid]`，不是 `preorder[1:mid]`。
- 空数组返回 `None`，这是终止条件。


## 解法名称

**递归分治（Recursive Divide and Conquer）**。

## 暴力思路

没有更简单的暴力做法，这题的关键就是“切”的规则。

## 优化思路

四步递归：

```text
mid = inorder.index(preorder[0])       # 根在中序里的位置
root = TreeNode(preorder[0])
root.left  = 用 preorder[1 : 1+mid] 和 inorder[:mid] 递归
root.right = 用 preorder[1+mid :]   和 inorder[mid+1:] 递归
```

时间：每一层要 `index` 一次，最坏 O(n)，总共 O(n²)。数据量 3000 时能过。如果面试官要求 O(n)，可以预先用一个字典把“值 -> 中序下标”存起来，这样查根就是 O(1)，总时间 O(n)。

空间：切片会产生新列表，加上递归栈，是 O(n²) 级别的额外空间；改成传下标范围可以降到 O(h)。教学版本先用切片，好理解。


## 你来写：从前序与中序遍历构造二叉树

要求：

- 用递归分治写，先找根，再切左右。
- 空数组返回 `None`。
- 写完用 `[3,9,20,15,7]` 和 `[9,3,15,20,7]` 建树，再用 `tree_values` 检查形状。

先在心里回答：为什么左子树的前序是 `preorder[1:1 + mid]`？


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


# 题目：从前序与中序遍历序列构造二叉树
# 解法：递归分治（Recursive Divide and Conquer）
# 输入：前序数组 preorder 和中序数组 inorder，长度 1 到 3000，元素互不相同。
# 目标：根据两个序列还原出原来的二叉树。
# 输出：返回这棵树的根节点。
# 注意：前序第一个是根；用 inorder.index 找到根的位置 mid；左子树前序是 preorder[1:1+mid]；空数组返回 None。


def build_from_preorder_inorder(preorder, inorder):
    # 在这里写你的代码
    pass


root = build_from_preorder_inorder([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])
print(tree_values(root))


In [ ]:
root = build_from_preorder_inorder([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])
print(tree_values(root))          # 期望 [3, 9, 20, None, None, 15, 7]

root2 = build_from_preorder_inorder([-1], [-1])
print(tree_values(root2))         # 期望 [-1]

root3 = build_from_preorder_inorder([1, 2], [2, 1])
print(tree_values(root3))         # 期望 [1, 2]

root4 = build_from_preorder_inorder([1, 2, 3], [3, 2, 1])
print(tree_values(root4))         # 期望 [1, 2, None, 3]


## 参考答案：从前序与中序遍历序列构造二叉树

```python
def build_from_preorder_inorder_answer(preorder, inorder):
    if len(preorder) == 0:
        return None

    root_val = preorder[0]
    root = TreeNode(root_val)
    mid = inorder.index(root_val)

    root.left = build_from_preorder_inorder_answer(preorder[1:1 + mid], inorder[:mid])
    root.right = build_from_preorder_inorder_answer(preorder[1 + mid:], inorder[mid + 1:])
    return root
```

面试表达：

我用递归分治。前序遍历的第一个元素一定是根，所以先拿它建根节点；然后在后序的中序数组里找到这个值的位置 `mid`，它左边就是左子树的所有节点，右边是右子树的所有节点；因为左子树有 `mid` 个节点，所以前序数组里从下标 1 开始的 `mid` 个元素属于左子树，剩下的属于右子树。左右两边递归处理。这个写法时间最坏 O(n²)，如果想优化到 O(n)，可以先用字典存下每个值在中序里的下标，避免每次查找。


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


def build_from_preorder_inorder_answer(preorder, inorder):
    if len(preorder) == 0:
        return None

    root_val = preorder[0]
    root = TreeNode(root_val)
    mid = inorder.index(root_val)

    root.left = build_from_preorder_inorder_answer(preorder[1:1 + mid], inorder[:mid])
    root.right = build_from_preorder_inorder_answer(preorder[1 + mid:], inorder[mid + 1:])
    return root


print(tree_values(build_from_preorder_inorder_answer([3, 9, 20, 15, 7], [9, 3, 15, 20, 7])))
print(tree_values(build_from_preorder_inorder_answer([-1], [-1])))
print(tree_values(build_from_preorder_inorder_answer([1, 2], [2, 1])))


## 题目 2 做题前先补：什么叫“路径”

这题里的**路径**有两个硬性规定，读题时必须抓住：

1. 必须**从上往下**走（只能从父节点到子节点），不能拐弯向上；
2. 起点不一定是根，终点不一定是叶子，任何节点到它下面某个节点都算。

所以“路径总和”这件事，本质是：**对每个节点当起点，往下试试能不能凑出目标和。**

再看一个小转化：从 A 到 B 的路径和，等于“从根到 B 的和”减去“从根到 A 的父节点的和”。这个思路是进阶做法，等下会提到。

先看清楚“以某个节点为起点”的下探过程。


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


def count_from(node, target):
    """以 node 为起点往下走，返回和为 target 的路径条数（带打印）。"""
    if node is None:
        return 0
    count = 0
    if node.val == target:
        count = 1
        print("    找到一个路径，终点是", node.val)
    count = count + count_from(node.left, target - node.val)
    count = count + count_from(node.right, target - node.val)
    return count


root = build_tree([10, 5, -3, 3, 2, None, 11, 3, -2, None, 1])
print("以根 10 为起点的路径数：", count_from(root, 8))


# 题目 2：路径总和 III

LeetCode 437. Path Sum III

## 题目描述（改写版）

给你一棵二叉树的根节点 `root` 和一个整数 `targetSum`，请统计树中**路径和等于 targetSum 的路径条数**。

路径的规定：

- 方向必须从上往下（父节点到子节点）；
- 起点可以是任意节点，终点也可以是任意节点；
- 路径至少包含一个节点。

## 输入

- `root`：二叉树根节点，节点个数 0 到 1000，节点值 -1000000000 到 1000000000。
- `targetSum`：目标整数。

## 输出

返回满足条件的路径条数。

## 示例

示例 1：树是 `[10,5,-3,3,2,None,11,3,-2,None,1]`，`targetSum = 8`，答案是 3。三条路径分别是 `5 -> 3`、`5 -> 2 -> 1`、`-3 -> 11`。

示例 2：树是 `[5,4,8,11,None,13,4,7,2,None,None,5,1]`，`targetSum = 22`，答案是 3。

## 易漏细节

- 路径不能拐弯，只能一路向下。
- 节点值可能是负数，所以不能“一旦超过目标就停”。
- 单节点路径也算，起点不要求是根。


## 解法名称

**双重递归（Double Recursion）**，也叫“以每个节点为起点的 DFS”。

## 暴力思路

就是上面的双重递归：

- 外层：对每个节点，都把它当作起点算一次；
- 内层：从起点往下走，沿途累加，看有没有凑到目标。

时间 O(n²)，每个节点都可能被上层的所有祖先访问一次；空间 O(h)。

## 优化思路

用前缀和的思想可以优化到 O(n)：一边 DFS 一边记录“从根到当前节点的路径和”，用字典统计每个和出现过几次；走到当前节点时，只要看“当前和 - targetSum”在字典里出现过几次，就说明有几条合法的路径。这题和 Day 03 的“和为 K 的子数组”是同一个思想在树上的版本。

数据量 1000 时双重递归完全够用，先把这个写熟，前缀和版本作为进阶。


## 你来写：路径总和 III

要求：

- 用双重递归写：外层遍历每个节点，内层从该节点往下找。
- 空节点返回 0。
- 写完用示例的两组数据各跑一遍。

先在心里回答：为什么节点值可以是负数时，不能“累加超过目标就提前返回”？


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


# 题目：路径总和 III
# 解法：双重递归（Double Recursion）
# 输入：二叉树根节点 root（节点数 0 到 1000），整数目标值 targetSum。
# 目标：统计所有从上往下、路径和等于 targetSum 的路径条数。
# 输出：返回条数（整数）。
# 注意：路径必须一路向下不能拐弯；起点不一定是根；节点值可能是负数，不能提前结束。


def path_sum(root, target_sum):
    # 在这里写你的代码
    pass


root = build_tree([10, 5, -3, 3, 2, None, 11, 3, -2, None, 1])
print(path_sum(root, 8))


In [ ]:
root = build_tree([10, 5, -3, 3, 2, None, 11, 3, -2, None, 1])
print(path_sum(root, 8))         # 期望 3

root2 = build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1])
print(path_sum(root2, 22))       # 期望 3

print(path_sum(build_tree([]), 1))          # 期望 0
print(path_sum(build_tree([1]), 1))         # 期望 1
print(path_sum(build_tree([1, 2]), 3))      # 期望 1
print(path_sum(build_tree([1, -2, -3]), -2))    # 期望 2（路径 1 -> -3，以及单独的 -2）


## 参考答案：路径总和 III

```python
def count_paths_from(node, target):
    if node is None:
        return 0
    count = 0
    if node.val == target:
        count = 1
    count = count + count_paths_from(node.left, target - node.val)
    count = count + count_paths_from(node.right, target - node.val)
    return count


def path_sum_answer(root, target_sum):
    if root is None:
        return 0
    total = count_paths_from(root, target_sum)
    total = total + path_sum_answer(root.left, target_sum)
    total = total + path_sum_answer(root.right, target_sum)
    return total
```

面试表达：

我用双重递归。内层函数 `count_paths_from` 负责“以某个节点为起点往下走”，每到一个节点就判断当前剩余目标是否正好等于它的值，然后带着减去当前值的目标继续往左右走。外层函数对每个节点都调用一次内层，这样所有起点都被覆盖到。时间 O(n²)，空间 O(h)。如果要求更快，可以用前缀和加字典做到 O(n)，思路和“和为 K 的子数组”一样。


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


def count_paths_from(node, target):
    if node is None:
        return 0
    count = 0
    if node.val == target:
        count = 1
    count = count + count_paths_from(node.left, target - node.val)
    count = count + count_paths_from(node.right, target - node.val)
    return count


def path_sum_answer(root, target_sum):
    if root is None:
        return 0
    total = count_paths_from(root, target_sum)
    total = total + path_sum_answer(root.left, target_sum)
    total = total + path_sum_answer(root.right, target_sum)
    return total


print(path_sum_answer(build_tree([10, 5, -3, 3, 2, None, 11, 3, -2, None, 1]), 8))    # 3
print(path_sum_answer(build_tree([5, 4, 8, 11, None, 13, 4, 7, 2, None, None, 5, 1]), 22))   # 3
print(path_sum_answer(build_tree([]), 1))      # 0
print(path_sum_answer(build_tree([1, 2]), 3))  # 1


## 题目 3 做题前先补：负数要不要

昨天做过“二叉树的直径”，用的是“左深度 + 右深度”。

今天这道题把深度换成了**路径上的数值和**，于是多了一个问题：**负数要不要？**

答案很直接：**如果某一边的贡献是负数，就不要它**。因为要的是最大路径和，带着一个负的子树只会让结果变小。所以：

```python
left = gain(node.left)
if left < 0:
    left = 0          # 负数不要，当作 0
```

第二个要理解的点：路径的形状。穿过某个节点的路径是“左边一段 + 自己 + 右边一段”，这两段各自只能是一条直链（不能再分叉），这也是为什么函数返回值只能带上“数值更大的那一边”。

第三点：答案的初始值不能设成 0，因为可能整棵树都是负数，此时答案应该是“最大的那个单独节点”。所以初始化成 `root.val`。


In [ ]:
# 看一个全是负数的例子：答案不是 0，而是最大的那个数
values = [-3, -1, -2]
best = values[0]
for value in values:
    if value > best:
        best = value
print("如果初始化成 0，会错误地输出 0")
print("正确做法是初始化成第一个值，结果是：", best)


# 题目 3：二叉树中的最大路径和

LeetCode 124. Binary Tree Maximum Path Sum

## 题目描述（改写版）

给你一棵二叉树的根节点 `root`，请返回树中**任意一条路径的最大路径和**。

路径的规定：

- 路径是一条从某个节点出发、沿着父子关系走出来的链；
- 同一条路径上每个节点最多出现一次；
- 路径可以拐弯：左子树 -> 某个节点 -> 右子树，但拐弯只能在最高点那一次。

## 输入

- `root`：二叉树根节点，节点个数 1 到 30000，节点值 -1000 到 1000。

## 输出

返回最大的路径和。

## 示例

示例 1：树是 `[1,2,3]`，最大路径是 `2 -> 1 -> 3`，和是 6。

示例 2：树是 `[-10,9,20,None,None,15,7]`，最大路径是 `15 -> 20 -> 7`，和是 42。

示例 3：树是 `[-3]`，最大路径和是 -3。（注意不是 0，因为路径至少要有一个节点。）

## 易漏细节

- 全是负数时，答案是最大的那个负数，不能返回 0。
- 负的子树贡献要舍弃，当作 0。
- 函数返回的是“能给父节点用的最大贡献”，不等于答案本身。


## 解法名称

**后序遍历 + 舍弃负数（Post-Order with Global Max）**。

## 暴力思路

枚举每两个节点，找出路径再求和。树上路径数量太多，完全不现实。

## 优化思路

和昨天的直径几乎一样，只是把“深度”换成“最大贡献”：

1. 递归拿到左、右子树能给的最大贡献 `left`、`right`；
2. **负数的贡献不要**：`left < 0` 就当 0，`right < 0` 也当 0；
3. 用 `node.val + left + right` 更新全局答案（这条路径以当前节点为拐弯点）；
4. 返回 `node.val + max(left, right)` 给父节点——因为父节点只能选一边接上去，不能两边都要。

时间 O(n)，每个节点访问一次；空间 O(h)。

## 画图跟踪

以 `[-10, 9, 20, None, None, 15, 7]` 为例：

| 节点 | left | right | 更新答案 | 返回 |
| --- | --- | --- | --- | --- |
| 9 | 0 | 0 | max(-, 9) | 9 |
| 15 | 0 | 0 | max(-, 15) | 15 |
| 7 | 0 | 0 | max(-, 7) | 7 |
| 20 | 15 | 7 | max(-, 42) = 42 | 20 + 15 = 35 |
| -10 | 9 | 35 | 42 更大，不变 | -10 + 35 = 25 |

答案 42。


## 你来写：二叉树中的最大路径和

要求：

- 用后序遍历 + 列表装答案写。
- 两边贡献为负时当 0；答案初始化成 `root.val`。
- 写完用 `[1,2,3]`、`[-10,9,20,None,None,15,7]`、`[-3]` 各跑一遍。

先在心里回答：函数返回值为什么只能取左右中较大的那个，不能两个都带上？


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


# 题目：二叉树中的最大路径和
# 解法：后序遍历 + 舍弃负数（Post-Order with Global Max）
# 输入：二叉树根节点 root，节点个数 1 到 30000，节点值 -1000 到 1000。
# 目标：求任意一条路径的最大路径和（路径可以在最高点拐一次弯）。
# 输出：返回最大路径和（整数）。
# 注意：答案初始化成 root.val，不能是 0；左右贡献为负时当 0；返回值只能带较大的一边。


def max_path_sum(root):
    # 在这里写你的代码
    pass


print(max_path_sum(build_tree([1, 2, 3])))


In [ ]:
print(max_path_sum(build_tree([1, 2, 3])))                        # 期望 6
print(max_path_sum(build_tree([-10, 9, 20, None, None, 15, 7])))  # 期望 42
print(max_path_sum(build_tree([-3])))                            # 期望 -3
print(max_path_sum(build_tree([2, -1])))                         # 期望 2
print(max_path_sum(build_tree([-2, -1])))                        # 期望 -1
print(max_path_sum(build_tree([1, -2, 3])))                      # 期望 4


## 参考答案：二叉树中的最大路径和

```python
def max_path_sum_answer(root):
    best = [root.val]

    def gain(node):
        if node is None:
            return 0
        left = gain(node.left)
        if left < 0:
            left = 0
        right = gain(node.right)
        if right < 0:
            right = 0

        total = node.val + left + right
        if total > best[0]:
            best[0] = total

        if left > right:
            return node.val + left
        return node.val + right

    gain(root)
    return best[0]
```

面试表达：

我用后序遍历。递归函数返回的是“从这个节点往下走，能给父节点提供的最大贡献”。在计算时，先递归拿到左右子树的贡献，如果某一边是负数就舍弃、当作 0，因为带上负数只会让和变小。然后用 `当前值 + 左贡献 + 右贡献` 去更新全局答案，这代表以当前节点为拐弯点的那条路径。但返回给父节点时只能选左右中更大的那一边，因为父节点到当前节点的路径不能分叉。答案初始化成根节点的值，避免全负数时错误地返回 0。时间 O(n)，空间 O(h)。


In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    index = 1
    while queue and index < len(values):
        node = queue.popleft()
        if index < len(values) and values[index] is not None:
            node.left = TreeNode(values[index])
            queue.append(node.left)
        index += 1
        if index < len(values) and values[index] is not None:
            node.right = TreeNode(values[index])
            queue.append(node.right)
        index += 1
    return root


def tree_values(root):
    """build_tree 的反向工具：按层序把树转回列表，末尾多余的 None 省略。"""
    if root is None:
        return []
    result = []
    queue = deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            result.append(None)
            continue
        result.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while len(result) > 0 and result[-1] is None:
        result.pop()
    return result


def max_path_sum_answer(root):
    best = [root.val]

    def gain(node):
        if node is None:
            return 0
        left = gain(node.left)
        if left < 0:
            left = 0
        right = gain(node.right)
        if right < 0:
            right = 0

        total = node.val + left + right
        if total > best[0]:
            best[0] = total

        if left > right:
            return node.val + left
        return node.val + right

    gain(root)
    return best[0]


print(max_path_sum_answer(build_tree([1, 2, 3])))                        # 6
print(max_path_sum_answer(build_tree([-10, 9, 20, None, None, 15, 7])))  # 42
print(max_path_sum_answer(build_tree([-3])))                            # -3
print(max_path_sum_answer(build_tree([2, -1])))                         # 2


# 今日小结

今天三个模式：

1. **构造二叉树**：前序定根，中序分左右，递归切分。
2. **路径总和**：外层遍历起点，内层往下累加；进阶用前缀和 + 字典优化到 O(n)。
3. **最大路径和**：后序遍历，负贡献舍弃，更新答案用两边，返回值只用一边，初始化成根节点值。

这三天（Day 13 到 Day 15）是本次冲刺里难度最高的部分，如果今天的三道题都自己写出来了，树的题型基本就过关了。


## 今日复盘区

- 构造二叉树时，左子树的前序为什么是 `preorder[1:1 + mid]`？
- 路径总和里，为什么要用“剩余目标”往下一层层减？
- 最大路径和里，为什么返回值只能带一边、答案却可以用两边？
- 全是负数时为什么答案不是 0？
- 今天哪几道题能不看答案写出来？

完成情况记录：

- 独立写出：
- 卡住的题：
- 明天重写：
- 完成日期：
